In [1]:
# !pip install evaluate
# !pip install rouge_score
# !pip install -U datasets
# conda install numpy pandas tqdm nltk jupyter
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# pip install transformers datasets accelerate evaluate sentencepiece
# pip install "protobuf>=3.19.6,<6" --upgrade

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, get_scheduler
from transformers import BartForConditionalGeneration, MBart50TokenizerFast
from datasets import load_dataset,load_from_disk

from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch

from accelerate import Accelerator
import evaluate

import wandb


In [2]:
import torch

print("PyTorch compiled with CUDA:", torch.version.cuda)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (compiled):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


PyTorch compiled with CUDA: 12.6
PyTorch version: 2.7.1+cu126
CUDA available: True
CUDA version (compiled): 12.6
cuDNN version: 90501
GPU name: NVIDIA A100-SXM4-40GB


In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dataset = load_dataset('/project/lt200246-mmacma/Big_seq2seq/data/text-to-gloss_ver2', cache_dir=None)

# Finetune model

In [4]:
model = '/project/lt200246-mmacma/Big_seq2seq/model/facebook/mbart-large-50'

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model)
tokenizer.src_lang = "th_TH"

HFValidationError: Repo id must use alphanumeric chars or '-', '_', '.', '--' and '..' are forbidden, '-' and '.' cannot start or end the name, max length is 96: 'MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250055, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250055, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_f

In [ ]:
#Test
input = tokenizer("และปิดท้ายกันที่กรุงเทพมหานครและปริมณฑล อุณหภูมิต่ำสุด 24 องศา สูงสุด 35 องศา มีฝนฟ้าคะนองร้อยละ 70 ของพื้นที่ค่ะ")
output = tokenizer("#c กรุงเทพมหานคร + จังหวัด + พื้นที่ใกล้เคียง(กรุงเทพมหานครปริมณฑล)|*วันนี้(/ช่วงนี้)|เย็น|อุณหภูมิต่ำ|#c 20+4(24)|ร้อน|อุณหภูมิสูง|แตะถึง|#c 30+5(35)|#c ฝนตก + ในหลายพื้นที่[มีทิศทางประกอบตั้งแต่ฝนตก ใช้สีหน้า 'ปานกลาง' เป็นตัวเชื่อมกับร้อยละของฝนในแต่ละพื้นที่ ใช้ทิศทางการเคลื่อนไหวรอบ ๆ]|*เปอร์เซ็นต์(/ร้อยละ)|70")
print("input part")
print(input)
print(tokenizer.convert_ids_to_tokens(input.input_ids))
print(tokenizer.decode(input.input_ids))
print("-----------------------------------------")
print("output part")
print(output)
print(tokenizer.convert_ids_to_tokens(output.input_ids))
print(tokenizer.decode(output.input_ids))

input part
{'input_ids': [250046, 1494, 34031, 96795, 3040, 699, 237798, 1213, 124515, 190899, 6, 186256, 90766, 15355, 744, 233470, 6, 54887, 2273, 233470, 5451, 83480, 56511, 44378, 2030, 24207, 219107, 2358, 16841, 26060, 11699, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['th_TH', '▁และ', 'ปิด', 'ท้าย', 'กัน', 'ที่', 'กรุงเทพมหานคร', 'และ', 'ปริ', 'มณฑล', '▁', 'อุณหภูมิ', 'ต่ํา', 'สุด', '▁24', '▁องศา', '▁', 'สูงสุด', '▁35', '▁องศา', '▁มี', 'ฝน', 'ฟ้า', 'คะ', 'น', 'อง', 'ร้อยละ', '▁70', '▁ของ', 'พื้นที่', 'ค่ะ', '</s>']
th_TH และปิดท้ายกันที่กรุงเทพมหานครและปริมณฑล อุณหภูมิต่ําสุด 24 องศา สูงสุด 35 องศา มีฝนฟ้าคะนองร้อยละ 70 ของพื้นที่ค่ะ</s>
-----------------------------------------
output part
{'input_ids': [250046, 468, 238, 119435, 997, 36039, 997, 6, 26060, 149781, 132, 237798, 124515, 190899, 16, 58745, 1639, 34316, 132, 64, 30387, 2417, 16, 58745, 41165, 58745, 186256, 90766, 58745, 4904, 238, 387, 13

### Add new tokenizer vocab

In [7]:
tokenizer.additional_special_tokens

['ar_AR',
 'cs_CZ',
 'de_DE',
 'en_XX',
 'es_XX',
 'et_EE',
 'fi_FI',
 'fr_XX',
 'gu_IN',
 'hi_IN',
 'it_IT',
 'ja_XX',
 'kk_KZ',
 'ko_KR',
 'lt_LT',
 'lv_LV',
 'my_MM',
 'ne_NP',
 'nl_XX',
 'ro_RO',
 'ru_RU',
 'si_LK',
 'tr_TR',
 'vi_VN',
 'zh_CN',
 'af_ZA',
 'az_AZ',
 'bn_IN',
 'fa_IR',
 'he_IL',
 'hr_HR',
 'id_ID',
 'ka_GE',
 'km_KH',
 'mk_MK',
 'ml_IN',
 'mn_MN',
 'mr_IN',
 'pl_PL',
 'ps_AF',
 'pt_XX',
 'sv_SE',
 'sw_KE',
 'ta_IN',
 'te_IN',
 'th_TH',
 'tl_XX',
 'uk_UA',
 'ur_PK',
 'xh_ZA',
 'gl_ES',
 'sl_SI']

In [8]:
def fix_tokenizer(tokenizer, new_lang='__thai_gloss__'):
    print('tokenizer len before: ', len(tokenizer))
    
    if new_lang not in tokenizer.additional_special_tokens:
        tokenizer.add_special_tokens({'additional_special_tokens': [new_lang]})

    token_id = tokenizer.convert_tokens_to_ids(new_lang)
    print(token_id)
    print('tokenizer len after: ', len(tokenizer))

In [9]:
fix_tokenizer(tokenizer, '__thai_gloss__')

tokenizer len before:  250054
250054
tokenizer len after:  250055


In [10]:
print(tokenizer.convert_ids_to_tokens([250051, 250052, 250053, 250054]))

['gl_ES', 'sl_SI', '<mask>', '__thai_gloss__']


### Preprocess data

In [5]:
input_language = "th_TH"
output_language = '__thai_gloss__'

tokenizer.src_lang = input_language
tokenizer.tgt_lang = output_language

In [6]:
#text, gloss_sequence, text_raw, text_sign

In [7]:
lens = [len(tokenizer.encode(g)) for g in dataset["train"]["text"]]
print(f"Thai Min: {min(lens)}, Avg: {sum(lens)/len(lens):.2f}, Max: {max(lens)}")

lens = [len(tokenizer.encode(g)) for g in dataset["train"]["gloss_sequence"]]
print(f"Sign Min: {min(lens)}, Avg: {sum(lens)/len(lens):.2f}, Max: {max(lens)}")

Thai Min: 5, Avg: 33.62, Max: 91
Sign Min: 7, Avg: 46.90, Max: 118


In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['no', 'text', 'gloss_sequence'],
        num_rows: 1397
    })
    eval: Dataset({
        features: ['no', 'text', 'gloss_sequence'],
        num_rows: 200
    })
    test: Dataset({
        features: ['no', 'text', 'gloss_sequence'],
        num_rows: 50
    })
})

In [9]:
max_input_length = 200
max_target_length = 200

def preprocess_function(examples):
    model_inputs = tokenizer(examples["text"],
                             max_length=max_input_length,
                             truncation=True)

    target_output = f'{output_language} {examples["gloss_sequence"]}'
    model_outputs = tokenizer(target_output,
                              max_length=max_target_length,
                              truncation=True)

    model_inputs["labels"] = model_outputs["input_ids"][1:]

    return model_inputs

x = preprocess_function(dataset['train'][0])
print(x)
print((tokenizer.decode(x['labels'])))

{'input_ids': [250046, 6, 222432, 6, 30387, 2417, 83480, 33413, 139090, 47635, 5451, 83480, 219107, 105593, 16841, 26060, 4454, 2469, 17140, 5597, 4365, 80139, 31266, 1547, 5803, 48333, 104354, 233387, 7034, 11699, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [250054, 6, 222432, 58745, 34316, 58745, 4904, 238, 83480, 31266, 1328, 35622, 18310, 9971, 1328, 1201, 16322, 26060, 58745, 104928, 126494, 24230, 58745, 1197, 58745, 8959, 58745, 2839, 58745, 2623, 58745, 83480, 31266, 44385, 58745, 48481, 58745, 4904, 71, 233387, 7034, 58745, 4904, 238, 83480, 31266, 1328, 1201, 16322, 26060, 58745, 2]}
__thai_gloss__ ภาคใต้|วันนี้|#cฝนตก+ลดน้อยลง+ในหลายพื้นที่|เปอร์เซ็นต์|30|ถึง|40|กับ|ฝนตกหนัก|ทะเล|#dอ่าวไทย|#cฝนตก+ในหลายพื้นที่|</s>


In [10]:
tokenized_dataset = dataset.map(preprocess_function)

Map:   0%|          | 0/1397 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [11]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['no', 'text', 'gloss_sequence', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1397
    })
    eval: Dataset({
        features: ['no', 'text', 'gloss_sequence', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['no', 'text', 'gloss_sequence', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 50
    })
})

### Model config

In [17]:
model = AutoModelForSeq2SeqLM.from_pretrained(model, device_map='auto')

In [18]:
added_token_id = tokenizer.convert_tokens_to_ids(output_language)
similar_lang_id = tokenizer.convert_tokens_to_ids(input_language)

#Add new language token
model.resize_token_embeddings(len(tokenizer))
model.model.shared.weight.data[added_token_id] = model.model.shared.weight.data[similar_lang_id]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [14]:
print(similar_lang_id)
print(added_token_id)
print(model.model.shared.weight.data[added_token_id])
print(model.model.shared.weight.data[similar_lang_id])

NameError: name 'similar_lang_id' is not defined

In [20]:
print(tokenizer.convert_ids_to_tokens([added_token_id - 5]))
print(model.model.shared.weight.data[added_token_id - 5])

['ur_PK']
tensor([-0.0406,  0.0272, -0.0004,  ...,  0.0102,  0.0484,  0.0120],
       device='cuda:3')


In [21]:
print(model.config)

MBartConfig {
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_bias_logits": false,
  "add_final_layer_norm": true,
  "architectures": [
    "MBartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "early_stopping": true,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "forced_eos_token_id": 2,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_length": 200,
  "max_position_embeddings": 1024,
  "model_type": 

In [22]:
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 2,
  "early_stopping": true,
  "eos_token_id": 2,
  "forced_eos_token_id": 2,
  "max_length": 200,
  "num_beams": 5,
  "pad_token_id": 1
}



### Data collator

In [12]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest", return_tensors='pt')

In [13]:
tokenized_dataset = tokenized_dataset.remove_columns(dataset['train'].column_names)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1397
    })
    eval: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 50
    })
})

In [14]:
features = [tokenized_dataset['train'][i] for i in range(2)]
data_collator(features)

{'input_ids': tensor([[250046,      6, 222432,      6,  30387,   2417,  83480,  33413, 139090,
          47635,   5451,  83480, 219107, 105593,  16841,  26060,   4454,   2469,
          17140,   5597,   4365,  80139,  31266,   1547,   5803,  48333, 104354,
         233387,   7034,  11699,      2,      1,      1,      1,      1,      1],
        [250046, 126217, 201896,   2386,  45934,  41165,   1201,  29317,  74795,
              6, 186256, 219523,  15550, 233470,   5451,  83480, 175942,  21814,
          15318,  29027,   5597,  83480,   3312,  31266,   4436,  29317, 145360,
           1037,  37895,      6, 186256,  90766,  15355,    953, 233470,      2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[250054,      6, 222432,  58745,  34316,  

### Data Loader

In [15]:
batch_size = 8

test_dataloader = DataLoader(tokenized_dataset["test"],
                             collate_fn=data_collator,
                             batch_size=batch_size)

### Train model

In [27]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

In [28]:
#Optimize
optimizer = AdamW(model.parameters(), lr=2e-5)

wandb.init(
    project="seq2seq-training-dataver5",
    name="mbart",
    mode="offline" 
)

In [29]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data5/mbart",
    save_strategy= "no",
    num_train_epochs=8,
    do_train=True,
    per_device_train_batch_size=4,
    gradient_checkpointing=True,
    gradient_accumulation_steps=8,

    learning_rate=5e-05,
    lr_scheduler_type="linear",

    do_eval=True,
    eval_strategy="epoch",
    per_device_eval_batch_size=4,

    logging_strategy="epoch",
    report_to="wandb"
)

In [30]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [31]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
1,2.034900,0.716222
2,0.594500,0.562032
3,0.467300,0.535810
4,0.388600,0.521992
5,0.325500,0.531658
6,0.269800,0.559070
7,0.222500,0.590899
8,0.189500,0.602237


TrainOutput(global_step=776, training_loss=0.5615744344966928, metrics={'train_runtime': 763.2303, 'train_samples_per_second': 32.347, 'train_steps_per_second': 1.017, 'total_flos': 2422182317850624.0, 'train_loss': 0.5615744344966928, 'epoch': 8.0})

In [32]:
trainer.save_model()

/lustrefs/disk/project/lt200246-mmacma/Big_seq2seq/mbart/env_mbart/lib/python3.10/site-packages/transformers/modeling_utils.py:3685: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


# Inference

In [4]:
model = AutoModelForSeq2SeqLM.from_pretrained("/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data2/mbart", device_map='auto')
tokenizer = AutoTokenizer.from_pretrained("/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data2/mbart")

In [16]:
text_input = []
gloss_translate = []
answer = []

model.to(device)
model.eval()

for batch in tqdm(test_dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        input_ids = batch["input_ids"]
        labels = batch["labels"]

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=batch["attention_mask"],
            max_new_tokens=300,
            length_penalty=0.6,
            early_stopping=True,
            num_beams=4,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5,
        )

    input_ids = input_ids.cpu().numpy()
    outputs = outputs.cpu().numpy()
    labels = labels.cpu().numpy()

    # Replace -100 in labels with pad_token_id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    translation = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    label = tokenizer.batch_decode(labels, skip_special_tokens=True)
    input = tokenizer.batch_decode(input_ids, skip_special_tokens=True)

    text_input.extend(input)
    gloss_translate.extend(label)
    answer.extend(translation)

result = pd.DataFrame({
    "text": text_input,
    "true_gloss": gloss_translate,
    "predicted_gloss": answer
})

  0%|          | 0/7 [00:00<?, ?it/s]

In [17]:
result

,text,true_gloss,predicted_gloss
0,ชลบุรี ระยอง จันทบุรี และตราด,ชลบุรี|ระยอง|จันทบุรี|ตราด|,ชลบุรี|ระยอง|จันทบุรี|ตราด|
1,ภาคกลางอุณหภูมิสูงขึ้น ประมาณ 1-2 องศาเซลเซียส...,ภาคกลาง|พื้นที่นี้|ร้อน|อุณหภูมิสูง|#c1-2|#c#s...,ภาคกลาง|ร้อน|อุณหภูมิสูง|เพิ่ม|#c1-2|#c#sO+C|เ...
2,(คุณสุนิดา) สวัสดีค่ะคุณผู้ชม มาตรวจสอบสภาพอาก...,ฉัน|สวัสดี|ดู|อากาศ|กับ|#sT|#sN|#sN|โลก|บอกเล่า|,ฉัน|สวัสดี|คนดู|วันนี้|อากาศ|กับ|#sT|#sN|#sN|โ...
3,ทะเลภาคตะวันออกวันนี้คลื่นต่ํากว่า 1 เมตร ห่าง...,ภาคตะวันออก|ทะเล|สูง|1|#sม|ต่ําลงมา|แต่|ห่างฝั...,ภาคตะวันออก|ทะเล|สูง|1|#sม|ต่ําลงมา|สมมติ|ห่าง...
4,กรมอุตุนิยมวิทยาคาดการณ์ 26-30 กันยายนนี้ ทั่ว...,#sก|#sร|#sม|CLสถานที่|อากาศ|ระบุ|วันนี้|#cเดือ...,#sก|#sร|#sม|CLสถานที่|อากาศ|ระบุ|#cเดือน+9|วัน...
5,แต่ว่าข้อดีก็มีนะคะ เพราะว่าฝนที่ตกลงมานั้นช่ว...,ข้อดี|ไหน|สมมติ|ฝนตกหนัก|ช่วย|ร้อน|ลดลง|ข้อดี|,แต่|ข้อดี|ไหน|ฝนตกหนัก|ช่วย|ร้อน|ในหลายพื้นที่...
6,ทีนี้เรามาตรวจสอบสภาพอากาศแบบรายภาคกันค่ะ,วันนี้|บอกเล่า|อากาศ|กับ|ภาคเหนือ|ภาคอีสาน|ภาค...,ฉัน|บอกเล่า|อากาศ|กับ|ภาคเหนือ|ภาคอีสาน|ภาคกลา...
7,(คุณสุนิดา) สวัสดีค่ะคุณผู้ชม มาตรวจสอบสภาพอาก...,ฉัน|สวัสดี|คน|ดู|ทุกท่าน|วันนี้|อากาศ|กับ|#sT|...,ฉัน|สวัสดี|คนดู|วันนี้|อากาศ|กับ|#sT|#sN|#sN|โ...
8,ส่วนภาคกลาง ช่วงเช้าเช้ายังมีอากาศเย็นสบาย อุณ...,พื้นที่นี้|ภาคกลาง|ตอนเช้า|เย็น|ลมเย็น|สบายๆ|อ...,ภาคกลาง|ตอนเช้า|เย็น|สบายดี|เย็น|อุณหภูมิต่ํา|...
9,ปิดท้ายกันที่กรุงเทพมหานครและปริมณฑลค่ะ วันนี้...,#cกรุงเทพมหานคร+จังหวัด+พื้นที่ใกล้เคียง|วันนี...,#cกรุงเทพมหานคร+จังหวัด+พื้นที่ใกล้เคียง|วันนี...


In [18]:
result.to_csv("/project/lt200246-mmacma/Big_seq2seq/transcript/dataset_ver2/mbart/mbart_dataver2.csv")